# Contextual Compression Retriever

The **Contextual Compression Retriever** in LangChain is an advanced retriever that improves retrieval quality by **compressing documents after retrieval** — keeping only the **relevant content** based on the user's query.

---

## ❓ Query

> "What is photosynthesis?"

---

## 📄 Retrieved Document (Traditional Retriever)

> "The Grand Canyon is a famous natural site.  
> Photosynthesis is how plants convert light into energy.  
> Many tourists visit every year."

---

## ❌ Problem

- The retriever returns the **entire paragraph**
- Only **one sentence** is actually relevant to the query
- The rest is **irrelevant noise** that:
  - Wastes context window  
  - May confuse the LLM  

---

## ✅ What Contextual Compression Retriever Does

Returns only the relevant part:

> "Photosynthesis is how plants convert light into energy."

---

## ⚙️ How It Works

1. **Base Retriever** (e.g., FAISS, Chroma) retrieves *N documents*  
2. A **compressor** (usually an LLM) is applied to each document  
3. The compressor keeps only the parts **relevant to the query**  
4. **Irrelevant content is discarded**  

---

## 📌 When to Use

- Your documents are **long and contain mixed information**
- You want to **reduce context length** for LLMs
- You want to **improve answer accuracy** in RAG pipelines  

---

## 🧠 Key Idea

> Retrieve broadly → then **compress intelligently**

In [9]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_chroma import Chroma
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
embeddings=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [4]:
# Recreate the document objects from the previous data
documents = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [5]:
vector_store=Chroma.from_documents(
    documents=documents,
    collection_name="cpr",
    persist_directory="my-chroma",
    embedding=embeddings
)

In [6]:
llm=ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

In [13]:
compressor_retriver=ContextualCompressionRetriever(
  base_compressor=LLMChainExtractor.from_llm(llm),
  base_retriever=vector_store.as_retriever(search_kwargs={"k": 5})
)

In [14]:
query="What is PhtotSynthesis?"
compressed_results=compressor_retriver.invoke(query)

In [15]:
for i,doc in enumerate(compressed_results):
    print(f"\n ---Result---")
    print(f"Content:- \n {doc.page_content}")
    


 ---Result---
Content:- 
 Photosynthesis is the process by which green plants convert sunlight into energy.

 ---Result---
Content:- 
 The chlorophyll in plant cells captures sunlight during photosynthesis.
